In [1]:
import torch
import re

def tokenize(text):
    """
    Separates words and punctuation into individual tokens using Regular Expressions.
    This ensures that the model sees 'dogs.' as ['dogs', '.'] instead of a single word.
    """
    return re.findall(r"[\w']+|[.,!?;]|<end>", text)

In [2]:
tokenize("Hello I am AI.")

['Hello', 'I', 'am', 'AI', '.']

In [3]:
with open("corpus.txt", "r") as f:
    sentences = [line.strip() for line in f if line.strip()]

In [4]:
print(len(sentences))
sentences[0]

50


'i like dogs.'

In [5]:
sentences = [s + " <end>" for s in sentences]
sentences[0]

'i like dogs. <end>'

In [6]:
# Build vocabulary: A unique set of all tokens seen in the training data
all_tokens = []
for s in sentences:
    all_tokens.extend(tokenize(s))

In [8]:
print(len(all_tokens))


350


In [11]:
vocab = sorted(set(all_tokens))
print(len(vocab))
vocab.append("<pad>")  # Special token for padding shorter sequences
vocab.append("<unk>")  # For out-of-vocabulary words

144


In [12]:
stoi = {w: i for i, w in enumerate(vocab)}
itos = {i: w for w, i in stoi.items()}

In [17]:
vocab_size = len(vocab)
pad_idx = stoi["<pad>"]
unk_idx = stoi["<unk>"] # Reference for unknown words


print(vocab_size)

146


In [18]:
# Convert sentences to token IDs (Numerical representations)
data_tokens = [[stoi[w] for w in tokenize(s)] for s in sentences]
print(data_tokens)


[[69, 75, 33, 0, 1], [143, 75, 21, 0, 1], [134, 78, 4, 0, 1], [21, 5, 28, 0, 1], [33, 5, 79, 0, 1], [122, 20, 111, 92, 122, 82, 0, 1], [122, 32, 105, 70, 122, 94, 0, 1], [13, 47, 70, 122, 110, 0, 1], [2, 113, 12, 109, 2, 115, 0, 1], [44, 120, 70, 122, 14, 133, 0, 1], [122, 11, 32, 7, 6, 122, 20, 0, 1], [122, 74, 96, 139, 142, 0, 1], [100, 36, 57, 56, 0, 1], [66, 104, 41, 70, 122, 43, 0, 1], [122, 118, 72, 16, 3, 132, 0, 1], [122, 85, 108, 6, 90, 0, 1], [69, 106, 2, 17, 32, 0, 1], [143, 106, 2, 137, 20, 0, 1], [134, 106, 122, 57, 126, 0, 1], [4, 89, 48, 3, 133, 0, 1], [122, 98, 72, 129, 60, 0, 1], [122, 125, 77, 70, 122, 49, 0, 1], [76, 5, 117, 3, 15, 0, 1], [37, 5, 129, 11, 0, 1], [84, 24, 122, 121, 126, 0, 1], [122, 101, 39, 92, 122, 46, 0, 1], [46, 58, 70, 122, 54, 0, 1], [10, 80, 119, 65, 0, 1], [122, 8, 111, 70, 2, 22, 0, 1], [93, 68, 6, 90, 0, 1], [2, 52, 73, 71, 122, 97, 0, 1], [122, 133, 72, 25, 3, 23, 0, 1], [69, 42, 122, 67, 13, 0, 1], [143, 95, 122, 114, 20, 0, 1], [134, 131,

In [19]:
# max_len defines the fixed sequence length for the model
max_len = max(len(tokens) for tokens in data_tokens)
max_len

9

In [20]:
def encode(s):
    """
    Converts a string of text into a list of token IDs.
    Uses .get() to default to <unk> if the word isn't in the vocab.
    """
    return [stoi.get(w, unk_idx) for w in tokenize(s)]

In [21]:
encode("i like dogs.")

[69, 75, 33, 0]

In [22]:
def decode(tokens):
    """Converts a list of token IDs back into a string, stopping at <end>."""
    words = []
    for t in tokens:
        w = itos[t]
        if w == "<end>":
            break
        words.append(w)
    return " ".join(words)

In [23]:
decode([69, 75, 33, 0])

'i like dogs .'

In [ ]:
def pad(seq):
    """
    Pads a sequence to max_len using the pad_idx.

    This ensures that all input tensors have the same shape, which is 
    required for efficient batch processing in GPUs/CPUs.
    """
    return seq + [pad_idx] * (max_len - len(seq))

In [26]:
pad([69, 75, 33, 0])

[69, 75, 33, 0, 144, 144, 144, 144, 144]

In [27]:
print(pad_idx)

144
